# Editor automático de videos — desde el navegador

No hace falta instalar nada en tu computadora: todo corre en los servidores de Google.
Funciona en cualquier PC, por vieja que sea, mientras el navegador abra esta página.

**Cómo se usa:** ejecuta las celdas en orden con el botón ▶ de la izquierda.
La primera tarda unos 2 minutos; las demás son rápidas.

Antes de empezar: arriba, en *Entorno de ejecución → Cambiar tipo de entorno*, deja **CPU**
(no hace falta GPU).

## 1. Instalar (2 minutos, una vez por sesión)

In [ ]:
#@title Instalar el editor { display-mode: "form" }
RAMA = "claude/auto-video-editing-app-rft92m" #@param {type:"string"}

!apt-get -qq install -y ffmpeg > /dev/null
![ -d /content/lab-editor-videos ] || git clone -q --branch $RAMA https://github.com/jona374/lab-editor-videos /content/lab-editor-videos
%cd /content/lab-editor-videos
!git fetch -q origin $RAMA && git checkout -q $RAMA && git pull -q
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, "/content/lab-editor-videos")
print("Listo. ffmpeg y el editor instalados.")

## 2. Tus claves (opcional)

- **Sin claves** igual puedes probar: pegas tú el guion y sale el video **sin voz**
  (con subtítulos y música). Sirve para ver el corte y el ritmo sin gastar nada.
- Con `ANTHROPIC_API_KEY` Claude escribe el guion solo.
- Con `ELEVENLABS_API_KEY` + `ELEVENLABS_VOICE_ID` se genera la voz en off.

Las claves se piden ocultas y **no quedan guardadas** en ningún lado: se borran al cerrar la pestaña.

In [ ]:
#@title Pegar claves (deja en blanco las que no tengas) { display-mode: "form" }
from getpass import getpass

claves = {
    "ANTHROPIC_API_KEY": getpass("ANTHROPIC_API_KEY (Enter para saltar): "),
    "ELEVENLABS_API_KEY": getpass("ELEVENLABS_API_KEY (Enter para saltar): "),
}
voz = input("ELEVENLABS_VOICE_ID (Enter para saltar): ").strip()
if voz:
    claves["ELEVENLABS_VOICE_ID"] = voz

import os
for nombre, valor in claves.items():
    if valor.strip():
        os.environ[nombre] = valor.strip()

print("Guion con Claude:", "sí" if os.getenv("ANTHROPIC_API_KEY") else "no (pega tú el guion)")
print("Voz en off:", "sí" if os.getenv("ELEVENLABS_API_KEY") else "no (video sin voz)")

## 3. Subir tus clips

Al ejecutar la celda aparece un botón **Elegir archivos**: selecciona los videos crudos
(puedes marcar varios a la vez). Si son pesados, tarda lo que tarde tu internet en subirlos.

In [ ]:
#@title Subir clips { display-mode: "form" }
MARCA = "textiles-pelileo" #@param {type:"string"}
TOMAS_DE_APOYO = False #@param {type:"boolean"}

import shutil
from pathlib import Path
from google.colab import files
from editor import marcas

marca = marcas.obtener(MARCA, crear=True)
destino = marca.tomas_de_apoyo if TOMAS_DE_APOYO else marca.por_editar

subidos = files.upload()
for nombre in subidos:
    shutil.move(nombre, destino / Path(nombre).name)

print(f"\n{len(marca.clips())} clip(s) por editar, {len(marca.clips_de_apoyo())} toma(s) de apoyo")

## 4. Música (opcional)

Sube uno o varios mp3. Se guardan en `assets/musica/<ambiente>/`, y el guion elige
el ambiente que le pega. Si no subes nada, el video sale sin música.

In [ ]:
#@title Subir música { display-mode: "form" }
AMBIENTE = "energico" #@param ["energico", "inspirador", "calmado", "urbano", "elegante"]

import shutil
from pathlib import Path
from google.colab import files

carpeta = Path("/content/lab-editor-videos/assets/musica") / AMBIENTE
carpeta.mkdir(parents=True, exist_ok=True)
for nombre in files.upload():
    shutil.move(nombre, carpeta / Path(nombre).name)

print("Pistas disponibles:")
!ls -R /content/lab-editor-videos/assets/musica

## 5. Hacer el video

Llena **Tema** (si tienes clave de Claude) **o** pega tu guion en **Guion**.
Si llenas los dos, manda el guion.

In [ ]:
#@title Generar { display-mode: "form" }
TEMA = "" #@param {type:"string"}
GUION = "" #@param {type:"string"}
DURACION = 46 #@param [30, 46, 60, 90] {type:"raw"}
CORTE_CADA = 2 #@param [1.5, 2, 3] {type:"raw"}
FORMATO = "vertical" #@param ["vertical", "cuadrado", "horizontal"]
SIN_VOZ = True #@param {type:"boolean"}
SUBTITULOS = True #@param {type:"boolean"}

from editor.config import Ajustes
from editor.pipeline import crear_video

ajustes = Ajustes(
    marca=MARCA,
    duracion_objetivo=float(DURACION),
    duracion_corte=float(CORTE_CADA),
    formato=FORMATO,
    subtitulos=SUBTITULOS,
)
resultado = crear_video(
    ajustes,
    tema=TEMA or None,
    texto_guion=GUION or None,
    sin_voz=SIN_VOZ,
    avisar=print,
)
print("\nVideo:", resultado.video)

## 6. Ver y descargar el resultado

In [ ]:
#@title Ver el video aquí mismo { display-mode: "form" }
import base64
from IPython.display import HTML

datos = base64.b64encode(resultado.video.read_bytes()).decode()
HTML(f'<video width="320" controls src="data:video/mp4;base64,{datos}"></video>')

In [ ]:
#@title Descargar el video y el guion { display-mode: "form" }
from google.colab import files

files.download(str(resultado.video))
files.download(str(resultado.carpeta / "guion.md"))
if resultado.subtitulos:
    files.download(str(resultado.subtitulos))

---

### Si algo falla

| Dice | Qué pasa |
|---|---|
| `no hay clips en …` | falta ejecutar la celda 3 (subir clips) |
| `falta ANTHROPIC_API_KEY` | pega el guion en **GUION** en vez de usar **TEMA** |
| `falta ELEVENLABS_API_KEY` | deja **SIN_VOZ** marcado |
| se reinició y perdió todo | Colab se apaga solo tras un rato inactivo: vuelve a correr desde la celda 1 |

Para hacer otro video con los mismos clips: cambia el tema en la celda 5 y ejecútala otra vez.